# 💻 IntelliCode-SL | Coding SLM — 8-bit Quantized (HuggingFace)

**Model:** `Qwen2.5-Coder-3B-Instruct`  
**Config:** 8-bit quantized base + full precision adapter (via HuggingFace BitsAndBytes)

> ⚠️ Uses HuggingFace directly — NOT Unsloth — because Unsloth's `load_in_8bit` has a  
> compiler bug with Qwen2 models (`BlockDiagonalCausalMask` NameError).  
> The adapter and test suite are identical to 07A.

Run **07A** first to get the 4-bit results, then run this notebook and compare.

> T4 GPU runtime required. Run cells top to bottom.

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────
!pip install -q transformers peft accelerate bitsandbytes
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────
# NOTE: Unsloth is intentionally NOT imported here.
# Loading 8-bit via Unsloth causes a BlockDiagonalCausalMask NameError on Qwen2.
import os, torch, time, gc, json
from collections import defaultdict
from google.colab import drive
from huggingface_hub import login
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

print("✅ Imports done")
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

In [ ]:
# ── Cell 3: Config + Drive + Login ─────────────────────────────
drive.mount("/content/drive")

HF_TOKEN     = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"  # ← paste your token
ADAPTER_PATH = "/content/drive/MyDrive/IntelliCode-SL/adapters/coding_adapter"
MODEL_NAME   = "Qwen/Qwen2.5-Coder-3B-Instruct"
MAX_SEQ_LEN  = 2048
TASKS        = ["debug", "generate", "modify"]

login(token=HF_TOKEN)
print("✅ Drive mounted + HF login done")
print(f"   Adapter: {ADAPTER_PATH}")

In [ ]:
# ── Cell 4: Prompt Templates ───────────────────────────────────
TASK_PROMPTS = {
    "debug"    : "Fix the bug in the following code and return the corrected version.",
    "generate" : "Write code based on the following description.",
    "modify"   : "Modify the following code according to the given instruction.",
}

PROMPT_TEMPLATE = """### Task: {task_instruction}

### Input:
{input}

### Output:
"""

print("✅ Prompt templates defined")

In [ ]:
# ── Cell 5: Test Cases (50 per task = 150 total) ───────────────
# Identical to 07A for fair comparison
test_cases = {
    "debug": [
        {"input": "def get_last(lst):\n    return lst[len(lst)]", "expected_fix": "lst[-1] or lst[len(lst)-1]"},
        {"input": "def count(s):\n    total = 0\n    for i in range(1, len(s)):\n        total += 1\n    return total", "expected_fix": "range(0, len(s))"},
        {"input": "def sum_list(nums):\n    total = 0\n    for i in range(1, len(nums)):\n        total += nums[i]\n    return total", "expected_fix": "range(0, len(nums))"},
        {"input": "def mid(lst):\n    return lst[len(lst)/2]", "expected_fix": "integer division //"},
        {"input": "nums = [1,2,3]\nfor i in range(1, len(nums)+1):\n    print(nums[i])", "expected_fix": "range(0, len(nums))"},
        {"input": "def subtract(a, b):\n    return a + b", "expected_fix": "a - b"},
        {"input": "def is_even(n):\n    return n % 2 == 1", "expected_fix": "n % 2 == 0"},
        {"input": "def area(w, h):\n    return w + h", "expected_fix": "w * h"},
        {"input": "def power(b, e):\n    return b + e", "expected_fix": "b ** e"},
        {"input": "def is_positive(n):\n    return n < 0", "expected_fix": "n > 0"},
        {"input": "def square(n):\n    result = n * n", "expected_fix": "add return result"},
        {"input": "def greet(name):\n    msg = f'Hello {name}'", "expected_fix": "add return msg"},
        {"input": "def double(x):\n    x = x * 2", "expected_fix": "add return x"},
        {"input": "def factorial(n):\n    if n == 0:\n        return 0\n    return n * factorial(n-1)", "expected_fix": "return 1 for base case"},
        {"input": "def fib(n):\n    if n <= 0:\n        return n\n    return fib(n-1) + fib(n-2)", "expected_fix": "if n <= 1"},
        {"input": "age = input('age: ')\nif age > 18:\n    print('adult')", "expected_fix": "int(input(...))"},
        {"input": "def add(a, b):\n    return a + b\n\nadd('5', 3)", "expected_fix": "int(a) + int(b) or type check"},
        {"input": "name = 'Alice'\nage = 30\nprint('Name: ' + name + ' Age: ' + age)", "expected_fix": "f-string or str(age)"},
        {"input": "def price(p):\n    return '$' + p", "expected_fix": "f'${p}' or str conversion"},
        {"input": "result = 7 / 2\nprint(result)  # expected 3", "expected_fix": "7 // 2"},
        {"input": "def get_name(d):\n    return d['name']\n\nget_name({'age': 25})", "expected_fix": "d.get('name')"},
        {"input": "scores = {'a': 1}\nprint(scores['b'])", "expected_fix": "scores.get('b', default)"},
        {"input": "def upper(s):\n    return s.upper()\n\nupper(None)", "expected_fix": "None check"},
        {"input": "def update(d, k):\n    d[k] += 1\n\nupdate({}, 'x')", "expected_fix": "d.get(k, 0) + 1"},
        {"input": "cfg = {'host': 'localhost'}\nport = cfg['port']", "expected_fix": "cfg.get('port', 8080)"},
        {"input": "def countdown(n):\n    while n > 0:\n        print(n)", "expected_fix": "add n -= 1"},
        {"input": "i = 0\nwhile i < 10:\n    print(i)", "expected_fix": "add i += 1"},
        {"input": "def repeat(s, n):\n    r = ''\n    while n > 0:\n        r += s\n    return r", "expected_fix": "add n -= 1"},
        {"input": "x = 10\nwhile x != 0:\n    x -= 3", "expected_fix": "x > 0"},
        {"input": "def find(lst, v):\n    i = 0\n    while lst[i] != v:\n        i += 0\n    return i", "expected_fix": "i += 1"},
        {"input": "def append_item(item, lst=[]):\n    lst.append(item)\n    return lst", "expected_fix": "lst=None, then if None: lst=[]"},
        {"input": "def add_tag(tag, tags={}):\n    tags[tag] = True\n    return tags", "expected_fix": "tags=None, then if None: tags={}"},
        {"input": "def avg(nums):\n    return sum(nums) / len(nums)", "expected_fix": "check if not nums"},
        {"input": "def pct(part, total):\n    return (part / total) * 100", "expected_fix": "check total == 0"},
        {"input": "total = 0\ndef add(n):\n    total += n\n    return total", "expected_fix": "global total"},
        {"input": "def make_counter():\n    count = 0\n    def inc():\n        count += 1\n        return count\n    return inc", "expected_fix": "nonlocal count"},
        {"input": "def rm_evens(nums):\n    for n in nums:\n        if n % 2 == 0:\n            nums.remove(n)\n    return nums", "expected_fix": "list comprehension"},
        {"input": "a = [1,2,3]\nb = a\nb.append(4)", "expected_fix": "b = a.copy()"},
        {"input": "def valid_age(age):\n    return age > 0 or age < 150", "expected_fix": "use 'and'"},
        {"input": "def can_vote(age, citizen):\n    return age >= 18 or citizen", "expected_fix": "use 'and'"},
        {"input": "def safe_div(a, b):\n    try:\n        return a / b\n    except:\n        pass", "expected_fix": "except ZeroDivisionError, return None"},
        {"input": "def parse(s):\n    try:\n        return int(s)\n    except ValueError as e:\n        print(e)", "expected_fix": "return default value"},
        {"input": "def is_palindrome(s):\n    return s == s[::-2]", "expected_fix": "s[::-1]"},
        {"input": "words = 'hello world'\nprint(words.split()[0].upper)", "expected_fix": "upper() with parentheses"},
        {"input": "class Dog:\n    def __init__(name, breed):\n        self.name = name", "expected_fix": "add self as first param"},
        {"input": "class Counter:\n    count = 0\n    def inc(self):\n        count += 1", "expected_fix": "self.count"},
        {"input": "def read(path):\n    f = open(path, 'r')\n    content = f.read()\n    return content", "expected_fix": "use with open(...) as f:"},
        {"input": "def sort_desc(lst):\n    return sorted(lst)", "expected_fix": "reverse=True"},
        {"input": "def most_freq(lst):\n    counts = {}\n    for x in lst:\n        counts[x] = counts.get(x,0)+1\n    return max(counts)", "expected_fix": "max(counts, key=counts.get)"},
        {"input": "def merge(a, b):\n    r = []\n    i = j = 0\n    while i < len(a) or j < len(b):\n        if a[i] < b[j]:\n            r.append(a[i]); i+=1\n        else:\n            r.append(b[j]); j+=1\n    return r", "expected_fix": "use 'and' not 'or', extend remaining"},
    ],
    "generate": [
        {"input": "Write a function to check if a number is prime", "expected_contains": "def is_prime"},
        {"input": "Write a function to reverse a string without using slicing", "expected_contains": "def reverse"},
        {"input": "Write a function to find the factorial of a number using recursion", "expected_contains": "def factorial"},
        {"input": "Write a function to flatten a nested list", "expected_contains": "def flatten"},
        {"input": "Write a function to count word frequencies in a string", "expected_contains": "def"},
        {"input": "Implement a binary search function that works on a sorted list", "expected_contains": "def binary_search"},
        {"input": "Write a Stack class with push, pop, peek, and is_empty methods", "expected_contains": "class Stack"},
        {"input": "Write a function to check if a string has balanced brackets", "expected_contains": "def"},
        {"input": "Write a merge sort implementation", "expected_contains": "def merge"},
        {"input": "Write a function to find all permutations of a list", "expected_contains": "def"},
        {"input": "Write a decorator that times how long a function takes to run", "expected_contains": "def"},
        {"input": "Write a memoize decorator for caching function results", "expected_contains": "def memoize"},
        {"input": "Write a function to validate an email address using regex", "expected_contains": "import re"},
        {"input": "Implement a Queue class using two stacks", "expected_contains": "class Queue"},
        {"input": "Write a function that generates all subsets of a list", "expected_contains": "def"},
        {"input": "Write a function to implement quicksort", "expected_contains": "def quicksort"},
        {"input": "Implement a simple LRU cache using OrderedDict", "expected_contains": "OrderedDict"},
        {"input": "Write a function to detect a cycle in a linked list using Floyd's algorithm", "expected_contains": "def"},
        {"input": "Write a function to find the two numbers that sum to a target in O(n)", "expected_contains": "def"},
        {"input": "Write a context manager that measures execution time", "expected_contains": "contextmanager"},
        {"input": "Write a generator function that yields chunks of a list", "expected_contains": "yield"},
        {"input": "Write a function to implement the Sieve of Eratosthenes", "expected_contains": "def sieve"},
        {"input": "Write a thread-safe singleton class", "expected_contains": "class"},
        {"input": "Write a function to compute Levenshtein distance between two strings", "expected_contains": "def"},
        {"input": "Write a simple rate limiter class using a sliding window", "expected_contains": "class"},
        {"input": "Write a function to find the longest increasing subsequence", "expected_contains": "def"},
        {"input": "Implement a Trie data structure with insert and search methods", "expected_contains": "class Trie"},
        {"input": "Write a retry decorator with exponential backoff", "expected_contains": "def retry"},
        {"input": "Write a function to serialize and deserialize a binary tree", "expected_contains": "def"},
        {"input": "Write a function to find all pairs in a list that sum to a target", "expected_contains": "def"},
        {"input": "Write a function to rotate a matrix 90 degrees clockwise", "expected_contains": "def"},
        {"input": "Write a Caesar cipher encoder and decoder", "expected_contains": "def caesar"},
        {"input": "Write a function to run-length encode a string", "expected_contains": "def"},
        {"input": "Write a function to find the kth largest element in an array", "expected_contains": "def"},
        {"input": "Write a class for a min heap with push and pop operations", "expected_contains": "class"},
        {"input": "Write a function to check if a sudoku board is valid", "expected_contains": "def"},
        {"input": "Write a function to generate all valid parentheses combinations for n pairs", "expected_contains": "def"},
        {"input": "Write a function to find the shortest path in an unweighted graph using BFS", "expected_contains": "def"},
        {"input": "Write a doubly linked list class with append, prepend, and delete methods", "expected_contains": "class"},
        {"input": "Write a function to compute the power set of a list", "expected_contains": "def"},
        {"input": "Write a function to find the number of islands in a 2D grid", "expected_contains": "def"},
        {"input": "Write a function to implement topological sort on a DAG", "expected_contains": "def"},
        {"input": "Write a function to detect if two strings are anagrams", "expected_contains": "def"},
        {"input": "Write a function to find the maximum subarray sum using Kadane's algorithm", "expected_contains": "def"},
        {"input": "Write a function to implement the knapsack problem using dynamic programming", "expected_contains": "def"},
        {"input": "Write a simple event emitter class", "expected_contains": "class"},
        {"input": "Write a function to convert a decimal number to binary without using bin()", "expected_contains": "def"},
        {"input": "Write a function that checks if a number is a perfect square without using sqrt", "expected_contains": "def"},
        {"input": "Write a function to calculate the number of ways to climb n stairs taking 1 or 2 steps", "expected_contains": "def"},
        {"input": "Write a function to find all paths between two nodes in a graph", "expected_contains": "def"},
    ],
    "modify": [
        {"input": "Add type hints to this function:\ndef add(a, b):\n    return a + b", "expected_contains": "->"},
        {"input": "Add error handling to this function:\ndef divide(a, b):\n    return a / b", "expected_contains": "ZeroDivision"},
        {"input": "Add logging to this function:\ndef process(data):\n    result = [x*2 for x in data]\n    return result", "expected_contains": "logging"},
        {"input": "Convert this loop to a list comprehension:\ndef squares(nums):\n    result = []\n    for n in nums:\n        result.append(n**2)\n    return result", "expected_contains": "["},
        {"input": "Add input validation to this function:\ndef get_elem(lst, idx):\n    return lst[idx]", "expected_contains": "raise"},
        {"input": "Refactor to use a dictionary instead of if-elif:\ndef day_name(n):\n    if n==1: return 'Mon'\n    elif n==2: return 'Tue'\n    elif n==3: return 'Wed'\n    else: return 'Unknown'", "expected_contains": "dict"},
        {"input": "Add caching to this function:\ndef fib(n):\n    if n <= 1: return n\n    return fib(n-1) + fib(n-2)", "expected_contains": "lru_cache"},
        {"input": "Convert to async:\ndef fetch(url):\n    import requests\n    return requests.get(url).json()", "expected_contains": "async"},
        {"input": "Add retry logic to this API call:\ndef call_api(url):\n    import requests\n    return requests.get(url).json()", "expected_contains": "retry"},
        {"input": "Refactor this class to use properties:\nclass Circle:\n    def __init__(self, r):\n        self.radius = r\n        self.area = 3.14 * r**2", "expected_contains": "@property"},
        {"input": "Add context manager support to this class:\nclass DB:\n    def __init__(self):\n        self.conn = None\n    def connect(self): pass\n    def disconnect(self): pass", "expected_contains": "__enter__"},
        {"input": "Make this class thread-safe:\nclass Counter:\n    def __init__(self):\n        self.val = 0\n    def inc(self):\n        self.val += 1", "expected_contains": "Lock"},
        {"input": "Add pagination to this function:\ndef get_users(db):\n    return db.query('SELECT * FROM users')", "expected_contains": "page"},
        {"input": "Optimize this O(n^2) function:\ndef has_dups(lst):\n    for i in range(len(lst)):\n        for j in range(i+1, len(lst)):\n            if lst[i]==lst[j]: return True\n    return False", "expected_contains": "set"},
        {"input": "Add soft delete to this class:\nclass Repo:\n    def __init__(self, db):\n        self.db = db\n    def delete(self, id):\n        self.db.execute('DELETE FROM users WHERE id=?',(id,))", "expected_contains": "deleted_at"},
        {"input": "Refactor to use generators:\ndef read_file(path):\n    with open(path) as f:\n        lines = f.readlines()\n    return [l.strip() for l in lines]", "expected_contains": "yield"},
        {"input": "Add method chaining to this builder:\nclass QueryBuilder:\n    def __init__(self, table):\n        self.table = table\n        self.conditions = []\n    def where(self, c):\n        self.conditions.append(c)\n    def build(self):\n        return f'SELECT * FROM {self.table}'", "expected_contains": "return self"},
        {"input": "Extend cache with TTL support:\nclass Cache:\n    def __init__(self):\n        self._data = {}\n    def get(self, k): return self._data.get(k)\n    def set(self, k, v): self._data[k] = v", "expected_contains": "time"},
        {"input": "Add batch support to this sender:\ndef send_email(to, subject, body):\n    pass  # sends single email", "expected_contains": "list"},
        {"input": "Add schema validation to this function:\ndef create_user(data):\n    db.insert('users', data)\n    return data", "expected_contains": "valid"},
        {"input": "Add metrics tracking to this cache:\nclass SimpleCache:\n    def __init__(self): self._d = {}\n    def get(self, k): return self._d.get(k)\n    def set(self, k, v): self._d[k] = v", "expected_contains": "hit"},
        {"input": "Add observer support to this class:\nclass Inventory:\n    def __init__(self):\n        self.items = {}\n    def add(self, name, qty):\n        self.items[name] = self.items.get(name,0) + qty", "expected_contains": "observer"},
        {"input": "Add circuit breaker to this client:\nclass Client:\n    def __init__(self, url):\n        self.url = url\n    def call(self, endpoint):\n        import requests\n        return requests.get(f'{self.url}/{endpoint}').json()", "expected_contains": "circuit"},
        {"input": "Add multiple auth strategies to this API:\nclass API:\n    def __init__(self, token):\n        self.token = token\n    def request(self, endpoint):\n        import requests\n        return requests.get(endpoint).json()", "expected_contains": "auth"},
        {"input": "Convert to support multiple output formats:\ndef report(data, title):\n    lines = [f'# {title}']\n    for k,v in data.items():\n        lines.append(f'- {k}: {v}')\n    return '\n'.join(lines)", "expected_contains": "format"},
        {"input": "Add environment variable config to this class:\nclass Config:\n    HOST = 'localhost'\n    PORT = 8080", "expected_contains": "os.getenv"},
        {"input": "Add serialization to this model:\nclass User:\n    def __init__(self, id, name, email):\n        self.id = id\n        self.name = name\n        self.email = email", "expected_contains": "to_dict"},
        {"input": "Add async iteration to this loader:\nclass Loader:\n    def __init__(self, data, size=32):\n        self.data = data\n        self.size = size\n    def __iter__(self):\n        for i in range(0, len(self.data), self.size):\n            yield self.data[i:i+self.size]", "expected_contains": "__aiter__"},
        {"input": "Add structured logging to this setup:\nimport logging\ndef setup_logger(name):\n    logger = logging.getLogger(name)\n    logger.setLevel(logging.DEBUG)\n    return logger", "expected_contains": "json"},
        {"input": "Add plugin support to this processor:\nclass TextProcessor:\n    def __init__(self, text):\n        self.text = text\n    def process(self):\n        return self.text.strip().lower()", "expected_contains": "plugin"},
        {"input": "Add export to this pipeline:\nclass Pipeline:\n    def __init__(self, data):\n        self.data = list(data)\n    def filter(self, fn):\n        self.data = [x for x in self.data if fn(x)]\n        return self", "expected_contains": "json"},
        {"input": "Add compression support to this writer:\ndef write_file(path, content):\n    with open(path, 'w') as f:\n        f.write(content)", "expected_contains": "gzip"},
        {"input": "Add file watching to this manager:\nclass FileManager:\n    def __init__(self, directory):\n        self.directory = directory\n    def list(self):\n        import os\n        return os.listdir(self.directory)", "expected_contains": "watch"},
        {"input": "Refactor this to use dependency injection:\nclass UserService:\n    def __init__(self):\n        self.db = Database()\n    def get_user(self, id):\n        return self.db.find(id)", "expected_contains": "def __init__(self, db"},
        {"input": "Add rate limiting to this endpoint:\ndef get_data(request):\n    data = fetch_from_db()\n    return {'data': data}", "expected_contains": "rate"},
        {"input": "Add connection retry to this DB class:\nclass DB:\n    def __init__(self, host, port):\n        import psycopg2\n        self.conn = psycopg2.connect(host=host, port=port)", "expected_contains": "retry"},
        {"input": "Add report formatting to this generator:\ndef generate(data, title):\n    return f'Report: {title}\\n{data}'", "expected_contains": "format"},
        {"input": "Add support for multiple hash algorithms to this hasher:\ndef hash_password(password):\n    import hashlib\n    return hashlib.md5(password.encode()).hexdigest()", "expected_contains": "algorithm"},
        {"input": "Add request timeout to this HTTP client:\nclass HTTPClient:\n    def __init__(self, base_url):\n        self.base_url = base_url\n    def get(self, path):\n        import requests\n        return requests.get(f'{self.base_url}{path}').json()", "expected_contains": "timeout"},
        {"input": "Add event sourcing to this aggregate:\nclass Order:\n    def __init__(self, id):\n        self.id = id\n        self.status = 'draft'\n    def submit(self):\n        self.status = 'submitted'", "expected_contains": "event"},
        {"input": "Add role-based access to this service:\nclass DataService:\n    def __init__(self):\n        pass\n    def get_all(self):\n        return db.query('SELECT * FROM data')", "expected_contains": "role"},
        {"input": "Add internationalization to this formatter:\ndef format_date(dt):\n    return dt.strftime('%Y-%m-%d')", "expected_contains": "locale"},
        {"input": "Add SQL injection protection to this query builder:\ndef query(table, condition):\n    return f'SELECT * FROM {table} WHERE {condition}'", "expected_contains": "param"},
        {"input": "Refactor to use strategy pattern:\ndef sort_data(data, method):\n    if method == 'bubble':\n        return bubble_sort(data)\n    elif method == 'merge':\n        return merge_sort(data)\n    else:\n        return sorted(data)", "expected_contains": "strategy"},
        {"input": "Add lazy loading to this repository:\nclass ProductRepo:\n    def __init__(self, db):\n        self.db = db\n        self.products = self.db.query('SELECT * FROM products')", "expected_contains": "lazy"},
        {"input": "Add audit logging to this user service:\nclass UserService:\n    def create(self, data):\n        user = db.insert('users', data)\n        return user\n    def delete(self, id):\n        db.delete('users', id)", "expected_contains": "audit"},
        {"input": "Add health check to this service:\nclass Service:\n    def __init__(self, db, cache):\n        self.db = db\n        self.cache = cache\n    def run(self):\n        pass", "expected_contains": "health"},
        {"input": "Add feature flags to this processor:\nclass FeatureProcessor:\n    def process(self, data):\n        result = transform(data)\n        return result", "expected_contains": "flag"},
        {"input": "Add graceful shutdown to this server:\nclass Server:\n    def __init__(self, port):\n        self.port = port\n        self.running = False\n    def start(self):\n        self.running = True", "expected_contains": "signal"},
    ]
}

print(f"✅ Test cases ready")
for task, cases in test_cases.items():
    print(f"   {task:<10} : {len(cases)} cases")

In [ ]:
# ── Cell 6: Evaluation Helper ──────────────────────────────────
def run_inference(model, tokenizer, task, input_text, max_new_tokens=300):
    prompt = PROMPT_TEMPLATE.format(
        task_instruction=TASK_PROMPTS[task],
        input=input_text
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Output:")[-1].strip()

def evaluate(model, tokenizer, config_name):
    print(f"\n{'='*58}")
    print(f"  Evaluating: {config_name}")
    print(f"{'='*58}")

    all_results = []
    per_task = {task: {"pass": 0, "fail": 0, "total": 0, "times": []} for task in TASKS}
    total_time = 0
    idx = 0

    for task, cases in test_cases.items():
        print(f"\n  Running {task} ({len(cases)} cases)...")
        for case in cases:
            t0 = time.time()
            output = run_inference(model, tokenizer, task, case["input"])
            elapsed = time.time() - t0
            total_time += elapsed
            per_task[task]["times"].append(elapsed)
            per_task[task]["total"] += 1

            passed = len(output) > 20 and not output.strip().startswith("###")
            if passed:
                per_task[task]["pass"] += 1
            else:
                per_task[task]["fail"] += 1

            all_results.append({
                "task": task,
                "input": case["input"][:80],
                "output": output[:200],
                "passed": passed,
                "time_ms": elapsed * 1000
            })
            idx += 1
            if idx % 25 == 0:
                total_pass = sum(v["pass"] for v in per_task.values())
                print(f"  [{idx:>3}/150]  running pass rate: {total_pass/idx*100:.1f}%")

    total_pass = sum(v["pass"] for v in per_task.values())
    total_cases = sum(v["total"] for v in per_task.values())
    avg_ms = total_time / total_cases * 1000

    print(f"\n  ✅ Overall Pass Rate : {total_pass/total_cases*100:.2f}% ({total_pass}/{total_cases})")
    print(f"  ⏱  Avg latency       : {avg_ms:.1f} ms/prompt")
    print(f"\n  {'Task':<12} {'Pass':>6} {'Fail':>6} {'Total':>7} {'Rate':>8} {'Avg ms':>8}")
    print(f"  {'-'*50}")
    for task in TASKS:
        s = per_task[task]
        rate = s["pass"] / s["total"] * 100
        avg_t = sum(s["times"]) / len(s["times"]) * 1000
        print(f"  {task:<12} {s['pass']:>6} {s['fail']:>6} {s['total']:>7} {rate:>7.1f}% {avg_t:>7.1f}ms")

    return {
        "config": config_name,
        "pass_rate": total_pass / total_cases * 100,
        "total_pass": total_pass,
        "total": total_cases,
        "avg_time_ms": avg_ms,
        "per_task": {
            task: {
                "pass": per_task[task]["pass"],
                "fail": per_task[task]["fail"],
                "total": per_task[task]["total"],
                "pass_rate": per_task[task]["pass"] / per_task[task]["total"] * 100,
                "avg_time_ms": sum(per_task[task]["times"]) / len(per_task[task]["times"]) * 1000
            } for task in TASKS
        },
        "samples": all_results[:10]
    }

print("✅ Evaluation helper defined")

In [ ]:
# ── Cell 7: Load Model — 8-bit via HuggingFace BitsAndBytes ────
print("Loading 8-bit quantized model via HuggingFace BitsAndBytes...")

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    token               = HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token = HF_TOKEN
)
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    torch_dtype = torch.float16
)
model.eval()

print("✅ Model loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 8: Run Evaluation ─────────────────────────────────────
results = evaluate(model, tokenizer, "Coding SLM — 8-bit quantized (HuggingFace)")

In [ ]:
# ── Cell 9: Sample Output Inspection ──────────────────────────
print("Sample outputs (first 3 per task):\n")
shown = {t: 0 for t in TASKS}
for r in results["samples"]:
    task = r["task"]
    if shown[task] < 3:
        print(f"  [{task.upper()}]")
        print(f"  Input  : {r['input'][:70]}...")
        print(f"  Output : {r['output'][:150]}...")
        print(f"  Passed : {'✅' if r['passed'] else '❌'} | {r['time_ms']:.0f}ms")
        print()
        shown[task] += 1

In [ ]:
# ── Cell 10: Save Results + Final Comparison ───────────────────
save_path = "/content/drive/MyDrive/IntelliCode-SL/benchmarks/coding_slm_8bit.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"✅ Results saved → {save_path}")

# Load 4-bit results for comparison if available
path_4bit = "/content/drive/MyDrive/IntelliCode-SL/benchmarks/coding_slm_4bit.json"
try:
    with open(path_4bit) as f:
        res_4bit = json.load(f)

    print(f"\n{'='*60}")
    print(f"  FINAL COMPARISON: Coding SLM — 4-bit vs 8-bit")
    print(f"{'='*60}")
    print(f"  {'Metric':<28} {'4-bit':>12} {'8-bit':>12}")
    print(f"  {'-'*54}")
    print(f"  {'Overall Pass Rate':<28} {res_4bit['pass_rate']:>11.2f}% {results['pass_rate']:>11.2f}%")
    print(f"  {'Pass / Total':<28} {res_4bit['total_pass']}/{res_4bit['total']:>9} {results['total_pass']}/{results['total']:>9}")
    print(f"  {'Avg Latency':<28} {res_4bit['avg_time_ms']:>10.1f}ms {results['avg_time_ms']:>10.1f}ms")
    print(f"\n  Per-Task Pass Rate:")
    print(f"  {'Task':<12} {'4-bit':>10} {'8-bit':>10} {'Diff':>8}")
    print(f"  {'-'*43}")
    for task in TASKS:
        a = res_4bit["per_task"][task]["pass_rate"]
        b = results["per_task"][task]["pass_rate"]
        diff = b - a
        sym = "▲" if diff > 0 else ("▼" if diff < 0 else "=")
        print(f"  {task:<12} {a:>9.1f}% {b:>9.1f}% {sym}{abs(diff):>5.1f}%")

    acc_diff = results['pass_rate'] - res_4bit['pass_rate']
    speedup  = res_4bit['avg_time_ms'] / results['avg_time_ms']
    print(f"\n  Speed   : 8-bit is {speedup:.2f}x {'faster' if speedup > 1 else 'slower'} than 4-bit")
    print(f"  Accuracy: 8-bit is {abs(acc_diff):.2f}% {'better' if acc_diff > 0 else 'worse'} than 4-bit")
    print(f"\n{'='*60}")
    print(f"  VERDICT")
    print(f"{'='*60}")
    if abs(acc_diff) <= 3.0 and speedup >= 1.0:
        print(f"  Pass rate gap is negligible (<=3%) and 8-bit is faster.")
        print(f"  ✅ RECOMMENDATION: Use 8-bit for production.")
    elif abs(acc_diff) <= 3.0 and speedup < 1.0:
        inv = 1/speedup
        print(f"  Pass rate gap negligible but 4-bit is {inv:.2f}x faster.")
        print(f"  ✅ RECOMMENDATION: Use 4-bit — same quality, faster.")
    elif acc_diff > 3.0:
        print(f"  8-bit is noticeably better (+{acc_diff:.2f}%).")
        print(f"  ✅ RECOMMENDATION: Use 8-bit.")
    else:
        print(f"  4-bit is noticeably better (+{abs(acc_diff):.2f}%).")
        print(f"  ✅ RECOMMENDATION: Use 4-bit.")
    print(f"{'='*60}")

except FileNotFoundError:
    print("\n⚠️  07A results not found. Run 07A first to enable comparison.")